# 🚀 FPGA vs CPU: Sobel Edge Detection Benchmark on PYNQ

## ✅ Step 1: Import Libraries

In [ ]:
from pynq import Overlay, allocate
from pynq.lib.video import VideoMode
import cv2
import numpy as np
from matplotlib import pyplot as plt
import time
from PIL import Image

%matplotlib inline

## 📥 Step 2: Load Your Image

In [ ]:
# Load the test image (e.g. cybertruck.jpg)
img = cv2.imread("/home/xilinx/jupyter_notebooks/sobel/cybertruck.jpg", cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (256, 256))  # Must match expected Sobel IP size
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis('off')
plt.show()

## ⚙️ Step 3: Run Sobel on CPU (OpenCV)

In [ ]:
start_cpu = time.time()

sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
sobel_cpu = cv2.convertScaleAbs(np.sqrt(sobelx**2 + sobely**2))

end_cpu = time.time()

plt.imshow(sobel_cpu, cmap='gray')
plt.title("CPU Sobel Output")
plt.axis('off')
plt.show()

print(f"CPU Execution Time: {end_cpu - start_cpu:.4f} seconds")

## 🔌 Step 4: Load FPGA Overlay

In [ ]:
ol = Overlay("/home/xilinx/jupyter_notebooks/sobel/sobel.bit")
sobel_ip = ol.sobel_0  # Adjust to your IP name
dma = ol.axi_dma_0     # Adjust to your DMA name

print("Overlay and IP loaded!")

## 🚀 Step 5: Run Sobel on FPGA

In [ ]:
in_buffer = allocate(shape=(256*256,), dtype=np.uint8)
out_buffer = allocate(shape=(256*256,), dtype=np.uint8)

# Load grayscale image into input buffer
in_buffer[:] = img.flatten()

start_fpga = time.time()

dma.sendchannel.transfer(in_buffer)
dma.recvchannel.transfer(out_buffer)
dma.sendchannel.wait()
dma.recvchannel.wait()

end_fpga = time.time()

sobel_fpga = out_buffer.reshape((256, 256))

plt.imshow(sobel_fpga, cmap='gray')
plt.title("FPGA Sobel Output")
plt.axis('off')
plt.show()

print(f"FPGA Execution Time: {end_fpga - start_fpga:.4f} seconds")

## 📊 Step 6: Summary Comparison

In [ ]:
print("✅ CPU Time:  ", round(end_cpu - start_cpu, 4), "seconds")
print("✅ FPGA Time: ", round(end_fpga - start_fpga, 4), "seconds")